# Prompts por nivel + auditoría (usa `taco_candidates.parquet`)
Ubicación: `modeling/notebooks/prompts_por_nivel.ipynb` (reemplaza al anterior).

Lee solo las columnas necesarias del parquet de 1.47 GB, así no se llena la memoria.

In [1]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

DATA = Path("../data")
PARQUET = DATA / "processed" / "taco_candidates.parquet"
OUT = DATA / "processed" / "por_nivel"
OUT.mkdir(parents=True, exist_ok=True)

NIVELES = ["EASY", "MEDIUM", "MEDIUM_HARD", "HARD", "VERY_HARD", "UNKNOWN_DIFFICULTY"]

pf = pq.ParquetFile(PARQUET)
print("Filas:", pf.metadata.num_rows)
print("Columnas:")
for campo in pf.schema_arrow:
    print(" -", campo.name, "|", campo.type)

Filas: 18575
Columnas:
 - question | string
 - solutions | string
 - starter_code | string
 - input_output | string
 - difficulty | string
 - raw_tags | string
 - name | string
 - source | string
 - tags | string
 - skill_types | string
 - url | string
 - Expected Auxiliary Space | string
 - time_limit | string
 - date | string
 - picture_num | string
 - memory_limit | string
 - Expected Time Complexity | string
 - split | string
 - orig_idx | int64


## 1. Elegir columnas
Mira la lista de arriba. El código intenta adivinar los nombres; si alguno sale `None`, escríbelo a mano.

In [2]:
cols = pf.schema_arrow.names

def buscar(opciones):
    for o in opciones:
        if o in cols:
            return o
    return None

COL_PROMPT = buscar(["question", "prompt", "problem", "text"])
COL_NIVEL  = buscar(["difficulty", "level", "nivel"])
COL_FUENTE = buscar(["source", "fuente"])
COL_SPLIT  = buscar(["split"])
COL_ID     = buscar(["id", "idx", "index", "problem_id"])

# Si alguno es None, escríbelo aquí, por ejemplo: COL_PROMPT = "question"
print(dict(prompt=COL_PROMPT, nivel=COL_NIVEL, fuente=COL_FUENTE, split=COL_SPLIT, id=COL_ID))

{'prompt': 'question', 'nivel': 'difficulty', 'fuente': 'source', 'split': 'split', 'id': None}


## 2. Cargar solo lo necesario

In [3]:
usar = [c for c in [COL_ID, COL_SPLIT, COL_FUENTE, COL_NIVEL, COL_PROMPT] if c]
df = pd.read_parquet(PARQUET, columns=usar)
df = df.rename(columns={COL_PROMPT: "prompt", COL_NIVEL: "difficulty",
                        COL_FUENTE: "source", COL_SPLIT: "split", COL_ID: "id"})
df["prompt"] = df["prompt"].fillna("").astype(str)
df["n_chars"] = df["prompt"].str.len()
df["difficulty"] = pd.Categorical(df["difficulty"].astype(str).str.upper(),
                                  categories=NIVELES, ordered=True)
print(df.shape)
df.head()

(18575, 5)


,split,source,difficulty,prompt,n_chars
0,train,codeforces,HARD,There are $n$ candy boxes in front of Tania. T...,2629
1,train,aizu,UNKNOWN_DIFFICULTY,"If you visit Aizu Akabeko shrine, you will fin...",1409
2,train,codeforces,EASY,"You have a deck of $n$ cards, and you'd like t...",3046
3,train,aizu,UNKNOWN_DIFFICULTY,"The number obtained by multiplying 1 by 2, 3, ...",1186
4,train,geeksforgeeks,EASY,Given a number s(in string form). Find the Sma...,798


## 3. Auditoría

In [4]:
print("Total de prompts:", len(df))
print("Vacíos:", (df["n_chars"] == 0).sum())
print("Muy cortos (<50 caracteres):", (df["n_chars"] < 50).sum())
print("Texto repetido:", df["prompt"].duplicated().sum())
if "id" in df and "split" in df:
    print("Ids repetidos (split, id):", df.duplicated(["split", "id"]).sum())
print("\nPrompts por nivel:")
print(df["difficulty"].value_counts(dropna=False).sort_index())
print("\nLargo del prompt por nivel (caracteres):")
print(df.groupby("difficulty", observed=True)["n_chars"].describe().round(0))

Total de prompts: 18575
Vacíos: 0
Muy cortos (<50 caracteres): 38
Texto repetido: 239

Prompts por nivel:
difficulty
EASY                  7814
MEDIUM                2404
MEDIUM_HARD           2348
HARD                  1763
VERY_HARD              980
UNKNOWN_DIFFICULTY    3266
Name: count, dtype: int64

Largo del prompt por nivel (caracteres):
                     count    mean     std    min     25%     50%     75%  \
difficulty                                                                  
EASY                7814.0  1235.0   710.0   52.0   726.0  1085.0  1639.0   
MEDIUM              2404.0  1481.0   710.0  171.0   938.0  1326.0  1900.0   
MEDIUM_HARD         2348.0  1759.0   817.0  152.0  1182.0  1682.0  2205.0   
HARD                1763.0  1969.0   780.0  175.0  1434.0  1864.0  2370.0   
VERY_HARD            980.0  2128.0   873.0  473.0  1537.0  2017.0  2536.0   
UNKNOWN_DIFFICULTY  3266.0  1591.0  1025.0   29.0   910.0  1319.0  1993.0   

                        max  
diffic

In [5]:
if "source" in df:
    display(pd.crosstab(df["difficulty"], df["source"], margins=True))

source,aizu,atcoder,codechef,codeforces,codewars,geeksforgeeks,hackerearth,leetcode,All
difficulty,,,,,,,,,
EASY,0,404,835,2128,2104,1612,566,165,7814
MEDIUM,0,129,261,863,20,766,154,211,2404
MEDIUM_HARD,0,129,524,1282,41,149,17,206,2348
HARD,0,47,301,1399,0,0,16,0,1763
VERY_HARD,0,48,457,475,0,0,0,0,980
UNKNOWN_DIFFICULTY,1415,566,747,14,0,0,524,0,3266
All,1415,1323,3125,6161,2165,2527,1277,582,18575


In [6]:
# Casos para revisar a mano
raros = df[(df["n_chars"] < 50) | (df["prompt"].duplicated(keep=False) & (df["n_chars"] > 0))]
raros.head(30)

,split,source,difficulty,prompt,n_chars
12,train,codeforces,MEDIUM_HARD,For some reason in many American cartoons anvi...,1512
74,train,hackerearth,UNKNOWN_DIFFICULTY,A palindrome is a string that is the same whet...,1120
90,train,hackerearth,UNKNOWN_DIFFICULTY,You are situated in an N dimensional grid at p...,945
190,train,hackerearth,UNKNOWN_DIFFICULTY,You need to handle two extraordinarily large i...,756
221,train,codeforces,HARD,"After all the events in Orlando we all know, S...",2896
223,train,hackerearth,UNKNOWN_DIFFICULTY,"When the day begins, it's time to sing carols....",705
259,train,codeforces,MEDIUM_HARD,Once upon a time in the Kingdom of Far Far Awa...,2241
331,train,hackerearth,UNKNOWN_DIFFICULTY,Heading ##There are two integers A and B. You ...,605
355,train,hackerearth,UNKNOWN_DIFFICULTY,Rose loves to play games. This problem is abou...,1381
373,train,codeforces,HARD,International Abbreviation Olympiad takes plac...,1283


## 4. Guardar los prompts por nivel

In [7]:
EXCLUIR_UNKNOWN = True   # cámbialo a False si los quieres

base = df[df["n_chars"] >= 50].drop_duplicates("prompt")
if EXCLUIR_UNKNOWN:
    base = base[base["difficulty"] != "UNKNOWN_DIFFICULTY"]

for nivel, g in base.groupby("difficulty", observed=True):
    if len(g) == 0:
        continue
    ruta = OUT / f"{str(nivel).lower()}.jsonl"
    g.to_json(ruta, orient="records", lines=True, force_ascii=False)
    print(f"{nivel:12s} {len(g):6d} prompts -> {ruta}")

print("\nDescartados (vacíos, cortos, repetidos o unknown):", len(df) - len(base))

EASY           7785 prompts -> ..\data\processed\por_nivel\easy.jsonl
MEDIUM         2376 prompts -> ..\data\processed\por_nivel\medium.jsonl
MEDIUM_HARD    2305 prompts -> ..\data\processed\por_nivel\medium_hard.jsonl
HARD           1715 prompts -> ..\data\processed\por_nivel\hard.jsonl
VERY_HARD       966 prompts -> ..\data\processed\por_nivel\very_hard.jsonl

Descartados (vacíos, cortos, repetidos o unknown): 3428
